### Notebook scope

### Purpose
Compute EP100 wave-component relationships with O3, U, and FWD across cases.

### Scientific question
Does the 0008-01 EP100 to O3/vortex relation extend to other hindcasts?

### Method
EP100 is mean(-ep2), 100 hPa, 40-80N. Use Jan20-Feb10 for 0008-01 and lead-day 10-30 for other cases.

### Expected output
Selected scatter figures, all-case heatmaps, and correlation CSV tables.

### Interpretation
Robust W1+W2 correlations across cases support planetary-wave forcing as a common predictability source.

### Caveat
Later-initialized cases test post-initialization forcing rather than long-lead precursor skill.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Build EP100/O3/U/FWD case metrics

### Purpose
Assemble member-level metrics and case-level correlations.

### Scientific question
Which EP100 component best relates to later O3 RMSE and vortex pathway metrics?

### Method
For each case, compute EP100 all/wave1/wave2/rest/W1+W2, O3 RMSE against BWCN same-year reference, U60 means, and FWD if available.

### Expected output
outputs/tables/03_EP100_case_member_metrics.csv and correlation tables.

### Interpretation
Positive R with O3 RMSE means stronger upward wave activity corresponds to larger O3 forecast error for that case.

### Caveat
Cases with small n or missing U/FWD products produce NA correlations and are shown gray in heatmaps.


In [ ]:
inv = discover_hindcast_cases()
metrics = []
for case in inv.loc[inv["can_source_diagnose"], "case"]:
    try:
        metrics.append(case_source_table(case, "primary"))
    except Exception as exc:
        log_message(f"{case}: case_source_table failed in notebook 03: {exc}")
member_metrics = pd.concat(metrics, ignore_index=True) if metrics else pd.DataFrame()
member_csv = TAB_DIR / "03_EP100_case_member_metrics.csv"
member_metrics.to_csv(member_csv, index=False)
print(member_metrics.head())

targets = ["O3_RMSE", "U60N10_mean", "U60N50_mean", "FWD_DOY"]
rows = []
for case, sub in member_metrics.groupby("case") if not member_metrics.empty else []:
    for target in targets:
        if target not in sub:
            continue
        c = finite_corr(sub["EP100_wave1_plus_wave2"], sub[target])
        rows.append({"case": case, "relationship": f"W1+W2 vs {target}", **c})
correlations = pd.DataFrame(rows)
corr_csv = TAB_DIR / "03_EP100_O3_U_FWD_correlations_all_cases.csv"
correlations.to_csv(corr_csv, index=False)

selected = [c for c in ["0008-01", "0013-02", "0014-02", "0019-02"] if c in set(member_metrics.get("case", []))]
if not selected:
    fig = empty_figure("No selected cases have metrics", "EP100 W1+W2 vs O3 RMSE")
else:
    fig, axes = plt.subplots(1, len(selected), figsize=(5.0*len(selected), 4.7), constrained_layout=True, squeeze=False)
    for ax, case in zip(axes.ravel(), selected):
        sub = member_metrics[member_metrics["case"] == case].dropna(subset=["EP100_wave1_plus_wave2", "O3_RMSE"])
        ax.scatter(sub["EP100_wave1_plus_wave2"], sub["O3_RMSE"], s=55, edgecolor="black", color="tab:purple")
        c = finite_corr(sub["EP100_wave1_plus_wave2"], sub["O3_RMSE"])
        ax.set_title(f"{case}\nR={c['R']:.2f}, p={c['p']:.2g}, n={c['n']}")
        ax.set_xlabel("EP100 W1+W2")
        ax.set_ylabel("O3 RMSE (DU)")
savefig(fig, "03_EP100_W1W2_vs_O3_RMSE_selected_cases", notebook="03_EP100_wave_component_multicase.ipynb", scientific_question="Does EP100 W1+W2 explain O3 RMSE in selected cases?", variables_windows="EP100 W1+W2; 100 hPa; 40-80N; source window; O3 RMSE init-May30", interpretation="Consistent positive or negative slopes across cases indicate a transferable EP100-O3 pathway relation.", caveat="Selected cases may have different initialization lead times and should not all be interpreted as long-lead precursor tests.", csv_table=member_csv)
plt.show()

heat = correlations.pivot(index="case", columns="relationship", values="R") if not correlations.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(12, max(4, 0.35*len(heat)+2)), constrained_layout=True)
if heat.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No correlations", ha="center", va="center")
else:
    im = ax.imshow(heat.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(heat.shape[1]), heat.columns, rotation=45, ha="right")
    ax.set_yticks(range(heat.shape[0]), heat.index)
    for i, case in enumerate(heat.index):
        for j, col in enumerate(heat.columns):
            p = correlations[(correlations["case"] == case) & (correlations["relationship"] == col)]["p"]
            star = "*" if len(p) and p.iloc[0] < 0.05 else ""
            ax.text(j, i, f"{heat.iloc[i,j]:.2f}{star}" if np.isfinite(heat.iloc[i,j]) else "NA", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="R")
    ax.set_title("EP100 W1+W2 vs O3/U/FWD correlations")
savefig(fig, "03_EP100_W1W2_vs_O3_U_FWD_heatmap_all_cases", notebook="03_EP100_wave_component_multicase.ipynb", scientific_question="Which target pathway metrics correlate with EP100 W1+W2 across cases?", variables_windows="EP100 W1+W2; O3_RMSE; U60N10/50 mean; FWD", interpretation="Starred cells identify nominally significant member-wise relationships.", caveat="FWD from U60N50 is calculated locally when needed and can be NA if the threshold is never met.", csv_table=corr_csv)
plt.show()

### Wave-component relevance heatmap

### Purpose
Compare all EP100 wave components against O3 RMSE.

### Scientific question
Is W1+W2 dominance unique to 0008-01 or recurrent across hindcast cases?

### Method
For every case, compute R between O3 RMSE and all waves, wave1, wave2, W1+W2, rest/synoptic waves, plus R(all waves, W1+W2).

### Expected output
outputs/figures/03_wave_component_relevance_heatmap_all_cases.png/pdf and outputs/tables/03_wave_component_relevance_all_cases.csv.

### Interpretation
High all-vs-W1+W2 and stronger O3 links for W1/W2 than rest supports planetary-wave dominance.

### Caveat
Correlation sign may differ when strong wave forcing improves the forecast rather than increases RMSE; interpret with scatter context.


In [ ]:
rows = []
for case, sub in member_metrics.groupby("case") if not member_metrics.empty else []:
    component_cols = ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest"]
    for col in component_cols:
        c = finite_corr(sub[col], sub["O3_RMSE"])
        rows.append({"case": case, "metric": col.replace("EP100_", "") + " vs O3_RMSE", **c})
    c = finite_corr(sub["EP100_all_waves"], sub["EP100_wave1_plus_wave2"])
    rows.append({"case": case, "metric": "all_waves vs W1+W2", **c})
wave_rel = pd.DataFrame(rows)
wave_csv = TAB_DIR / "03_wave_component_relevance_all_cases.csv"
wave_rel.to_csv(wave_csv, index=False)
mat = wave_rel.pivot(index="case", columns="metric", values="R") if not wave_rel.empty else pd.DataFrame()
fig, ax = plt.subplots(figsize=(12.5, max(4, 0.35*len(mat)+2)), constrained_layout=True)
if mat.empty:
    ax.axis("off"); ax.text(0.5, 0.5, "No wave-component correlations", ha="center", va="center")
else:
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=45, ha="right")
    ax.set_yticks(range(mat.shape[0]), mat.index)
    for i, case in enumerate(mat.index):
        for j, col in enumerate(mat.columns):
            p = wave_rel[(wave_rel["case"] == case) & (wave_rel["metric"] == col)]["p"]
            star = "*" if len(p) and p.iloc[0] < 0.05 else ""
            val = mat.iloc[i, j]
            ax.text(j, i, f"{val:.2f}{star}" if np.isfinite(val) else "NA", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="R")
    ax.set_title("Wave-component relevance for O3 RMSE")
savefig(fig, "03_wave_component_relevance_heatmap_all_cases", notebook="03_EP100_wave_component_multicase.ipynb", scientific_question="Is W1+W2 dominance robust across hindcast cases?", variables_windows="EP100 components; 100 hPa; 40-80N; source windows; O3 RMSE init-May30", interpretation="High all-vs-W1+W2 and weak rest/synoptic correlations support planetary-wave uncertainty as the main source.", caveat="Cases with later initialization diagnose post-initialization forcing, not pre-initialization wave buildup.", csv_table=wave_csv)
plt.show()
write_figure_guide()